# 04 — Pseudo-labels + split étude

Sortie de `src/data/build_labels.py` :

1. Extraction NLP sur les 4 349 rapports non-gold → `data/labels/pseudo_labels.csv`
2. Règle « rapport globalement normal » → 0 fiables sur les cibles non mentionnées
   (corrige le biais *mentionné = présent* : sans ça, Synovitis gardait 96 % de positifs)
3. Masquage par confiance (< 0.70, Effusion < 0.80) → `NaN` = ignoré en loss
4. Plafonnement de prévalence à 0.65 par cible (masquage aléatoire de positifs)
5. Split 85/15 **au niveau étude**, stratifié langue × nb de positifs →
   `data/labels/train_val_labels.csv`

Les 58 gold ne sont **pas** dans le train : réserve de sanity-check.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent
labels = pd.read_csv(ROOT / 'data/labels/train_val_labels.csv')
TARGETS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
           'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
print(labels.shape)
labels.head()

In [ ]:
# Couverture et prévalences par cible (labels gardés)
stats = pd.DataFrame({
    'couverture': labels[TARGETS].notna().mean(),
    'prevalence': labels[TARGETS].mean(),
    'n_gardes': labels[TARGETS].notna().sum(),
}).round(3).sort_values('couverture', ascending=False)
display(stats)
full_na = labels[TARGETS].isna().all(axis=1).sum()
print(f'études sans aucun label (exclues du train) : {full_na}')

In [ ]:
# Split : tailles et prévalences train/val par cible
print(labels.split.value_counts().to_string())
comp = pd.DataFrame({
    'train': labels[labels.split=='train'][TARGETS].mean(),
    'val': labels[labels.split=='val'][TARGETS].mean(),
}).round(3)
display(comp)
# garde-fou : aucun chevauchement d'études, jamais de split par série
tr = set(labels[labels.split=='train'].StudyInstanceUID)
va = set(labels[labels.split=='val'].StudyInstanceUID)
assert not (tr & va), 'fuite train/val !'
print('pas de chevauchement train/val OK')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stats['couverture'].plot.barh(ax=axes[0], title='Couverture (labels gardés)')
stats['prevalence'].plot.barh(ax=axes[1], title='Prévalence (labels gardés)')
for ax in axes: ax.set_xlim(0, 1)
plt.tight_layout()